## Topic: Predicting Patients Length-of-Stay (LOS) in Hospitals
By Group 2: Shuyuan Wang, Luoze Guo, Beibei Le, Ping-Jui Chen, Chia-Chen Li

## Part 1: EDA(Exploratory Data Analysis)
### 1.1 Data cleaning

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('Topic3_healthcare_analytics_dataset.csv', low_memory=False)
#summary of columns, non-full counts, data types
df.info()
print("\ncolumn names: \n", df.columns.tolist())
#See missing values
print("\nmissing values: \n", df.isnull().sum()) 

In [ ]:
#MCAR proxy test
#Create missingness indicators
df['Stay_missing'] = df['Stay'].isnull().astype(int)
df['Bed_Grade_missing'] = df['Bed Grade'].isnull().astype(int)
df['City_Code_Patient_missing'] = df['City_Code_Patient'].isnull().astype(int)
#list of numeric columns to test
numeric_cols = df.select_dtypes(include='number').columns.drop(
    ['Stay_missing', 'Bed_Grade_missing', 'City_Code_Patient_missing'])
# Group means comparison
print("Checking Stay missingness pattern:")
print(df.groupby('Stay_missing')[numeric_cols].mean())

print("\nChecking Bed Grade missingness pattern:")
print(df.groupby('Bed_Grade_missing')[numeric_cols].mean())

print("\nChecking City_Code_Patient missingness pattern:")
print(df.groupby('City_Code_Patient_missing')[numeric_cols].mean())
# For each column, 0=not missing, 1=missing, we compare mean values of other numeric columns between: rows where the value is not missing(0) and rows where the value is missing(1).
# If means are similar, then missing completely at random(MCAR), if means are different, then missingness depends on other variables(MAR/MNAR)

In [ ]:
#Stay: Since Stay is our target, missing labels cannot be used for training. S0 we drop those rows, where 'Stay' is missing.
df.columns = df.columns.str.strip()
df = df.dropna(subset=['Stay'])
#MCAR test of Bed Grade interpretation:Hospital_code: 0: 18,33, 1:6,11, shows a difference, suggesting that the missingness might vary across hospitals. The differences in Hospital_code and Admission_Deposit indicates that missing Bed Grade is not perfectly MCAR. However, the differences across most features are small and there is no strong, systematic pattern in missingness. So mode imputation will not significantly affect overall distributions.
#MCAR test of City_Code_Patient: The MCAR proxy test for City_Code_Patient showed minor differences in features between missing and non-missing groups, suggesting that missingness is close to MCAR or weakly MAR. Given the low proportion of missing data and lack of strong patterns, mode imputation is an appropriate and efficient choice for handling missing values in this column.
#Bed Grade & City_Code_Patient: both represent categorical numeric labels, so fill the missing value with mode (most frequent value)
df['Bed Grade'] = df['Bed Grade'].fillna(df['Bed Grade'].mode()[0])
df['City_Code_Patient'] = df['City_Code_Patient'].fillna(df['City_Code_Patient'].mode()[0])
print('Before:', df.shape)
#Drop duplicates
df = df.drop_duplicates()
print('After:', df.shape)

In [ ]:
#Dropping irrelevant features, that do not carry predictive value for Length of Stay
df = df.drop(columns=['case_id','patientid','Hospital_code','Hospital_region_code','City_Code_Hospital','City_Code_Patient_missing','City_Code_Patient'], errors='ignore')
print(df.columns)

### 1.2 Feature Engineering

In [ ]:
#Identify all categorical, non-numerical columns:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print(categorical_cols)

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [ ]:
#Converting categorical columns into numeric columns
#labelencoder: Type of Admission, Severity of Illness, Age, Stay
#Onehot: Hospital_type_code, Department, Ward_Type, Ward_Facility_Code

#Determine what each numeric label corresponds to in original bins
#Encoding Stay_Category by converting objects to numerical labels

#Confirm what df['Stay'] currently contains:
#Create new column: Stay_Category for simplified modeling
print(df['Stay'].unique())

short_bins  = ['0-10', '11-20']
medium_bins = ['21-30', '31-40', '41-50']
long_bins   = ['51-60','61-70','71-80','81-90','91-100','More than 100 Days']

category_map = {v:0 for v in short_bins}
category_map.update({v:1 for v in medium_bins})
category_map.update({v:2 for v in long_bins})

df['Stay_Category'] = df['Stay'].map(category_map).astype(int)

stay_string_map = {
    '0-10': 0,
    '11-20': 1,
    '21-30': 2,
    '31-40': 3,
    '41-50': 4,
    '51-60': 5,
    '61-70': 6,
    '71-80': 7,
    '81-90': 8,
    '91-100': 9,
    'More than 100 Days': 10
}
df['Stay'] = df['Stay'].map(stay_string_map).astype(int)
#Test
print(df[['Stay','Stay_Category']].head(10))
print("Counts by category:\n", df['Stay_Category'].value_counts().sort_index())

In [ ]:
#Confirm what df['Stay'] currently contains:
#Create new column: Stay_Category for simplified modeling
print(df['Stay'].unique())
#The columns is encoded

In [ ]:
#Create Age_Group
age_string_mapping = {
    '0-10':   0,
    '11-20':  1,
    '21-30':  2,
    '31-40':  3,
    '41-50':  4,
    '51-60':  5,
    '61-70':  6,
    '71-80':  7,
    '81-90':  8,
    '91-100': 9
}
df['Age'] = df['Age'].map(age_string_mapping).astype(int)
#Test
print(df['Age'].unique())
print(df['Age'].value_counts().sort_index())

In [ ]:
#Label-encode ordinal columns
#Map Type of Admission by seriousness: Urgent < Trauma < Emergency
adm_map = {'Urgent': 0, 'Trauma': 1, 'Emergency': 2}
df['Type of Admission'] = df['Type of Admission'].map(adm_map).astype(int)

#Map Severity of Illness by severity: Minor < Moderate < Extreme
sev_map = {'Minor': 0, 'Moderate': 1, 'Extreme': 2}
df['Severity of Illness'] = df['Severity of Illness'].map(sev_map).astype(int)

#Test
print(df[['Type of Admission','Severity of Illness']].head())
print(df.dtypes[['Type of Admission','Severity of Illness']])

In [ ]:
#one-hot encode nominal columns
df_one_hot = df.copy()

onehot_cols = ['Hospital_type_code', 'Department', 'Ward_Type', 'Ward_Facility_Code']
ohe = OneHotEncoder(drop='first', sparse_output=False)
encoded = ohe.fit_transform(df_one_hot[onehot_cols])
encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(onehot_cols),
    index=df_one_hot.index
)

df_one_hot = pd.concat([
    df_one_hot.drop(columns=onehot_cols).reset_index(drop=True),
    encoded_df.reset_index(drop=True)
], axis=1)

#Test
print(df_one_hot.head())
print(df_one_hot.dtypes)

In [ ]:
# Normalizing Admission_Deposit to a standard range[0,1] without changing the original distribution shape
from sklearn.preprocessing import MinMaxScaler

In [ ]:
scaler = MinMaxScaler()
df_one_hot['Admission_Deposit'] = scaler.fit_transform(df[['Admission_Deposit']])
#Test
print(df_one_hot.head())

### 1.3 Defining features(X) and target(Y) (Data Preparation for Modeling)

In [ ]:
#Features(X): the input variable used by the model to learn patterns and make predictions
#We drop column 'Stay', so that the model can't cheat(no data leakage)
X = df_one_hot.drop(['Stay'], axis=1)
#Target(y): the output variable we want to predict using those features
y = df_one_hot['Stay_Category']

### 1.4 Data Splitting(Train-Test-Split)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

### 1.5 Visualization

In [ ]:
#for the visualization, we use another DataFrame that does not contain the OneHotEncoder.
from sklearn.preprocessing import LabelEncoder
#Converting categorical columns into numeric columns
df_visualization = df.copy()
le = LabelEncoder()
for col in categorical_cols:
    df_visualization[col] = le.fit_transform(df[col])
#Test

print(df_visualization.dtypes)

# Normalizing Admission_Deposit to a standard range[0,1] without changing the original distribution shape
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_visualization['Admission_Deposit'] = scaler.fit_transform(df_visualization[['Admission_Deposit']])
#Test
print(df_visualization.head())

### 1.5.1 Bar plot: Distribution of Stay_Category

In [ ]:
!pip install seaborn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize = (10,5))
df_visualization['Stay_Category'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribution of Stay Categories')
plt.xlabel('Stay Category(Encoded version)')
plt.ylabel('Number of patients')
plt.show()
# The bar chart shows the number of patients (y-axis) in each stay category (Short, Medium, Long), providing an overview of the class distribution within the dataset.
# This helps identify whether the dataset is balanced or imbalanced across the different stay categories, which is important for modeling, as significant imbalance may affect the model’s performance.

### 1.5.2 Count Plot: Correlation between Stay_Category and Severity of Illness

In [ ]:
plt.figure(figsize = (10,5))
sns.countplot(x='Severity of Illness', hue='Stay_Category', data=df_visualization)
plt.title('Correlation between Stay Categories snd Severity of Illness')
plt.xlabel('Severity of Illness(Encoded version)')
plt.ylabel('Counts')
plt.legend(title='Stay Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()
# The count plot illustrates the distribution of stay categories (Short, Medium, Long) within each encoded illness severity level (0 = Mild, 1 = Moderate, 2 = Severe)
# It visually confirms the correlation between illness severity and the length of stay.
# From the plot, we can identify which illness severity levels tend to have longer lengths of stay by observing the proportion of Long stay categories across severity levels.

### 1.5.3 Box Plot: Correlation between Admission_Deposit and Stay_Category

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='Stay_Category', y='Admission_Deposit', data=df_visualization)
plt.title('Correlation between Admission Deposit and Stay Category')
plt.xlabel('Stay Category (Encoded)')
plt.ylabel('Normalized Admission Deposit')
plt.show()
# This box plot displays the distribution of normalized admission deposits across each stay category (Short, Medium, Long).
# It shows the middle value, spread, and any extreme deposit values in each stay type.
# Middle line(Median): shows the middle deposit value for that stay category.
# Box(IQR): shows where the middle 50% of deposit values lie for that stay category.
# Whiskers(lines extending from the box): shows tje overall spread of deposits, excluding outliers.
# Dots outside whiskers(Outliers): shows unusually high or low deposit values.
# The boxes look similar across categories: Medians are around 0.3 for all. The spread (IQR) and whiskers are nearly the same. Many outliers (high deposits) are present in all categories.
# Admission deposits do not vary significantly across stay lengths. Higher deposits do not clearly indicate longer stays in this dataset. This feature may have limited predictive power for Length of Stay.

### 1.5.4 Heatmap: Correlation Matrix

In [ ]:
#Drop the columns we don't need before generating the heatmap
cols_to_drop = ['Stay_missing','Bed_Grade_missing']
plt.figure(figsize=(12, 10))
sns.heatmap(df_visualization.drop(columns=cols_to_drop).corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()
# The heatmap shows correlation values between all numerical features in the dataset, including the target Stay_Category. Darker colors indicate stronger correlations (positive or negative), while lighter colors indicate weaker correlations.
# 	1: Strong positive relationship
# 	-1: Strong negative relationship
# 	0: No linear relationship
# The heatmap helps us see which features are related to each other and to Stay_Category, guiding feature selection and understanding relationships before modeling.
# Stay has a strong positive correlation (~0.87) with Stay_Category, which is expected since both represent length of stay.
# Severity of Illness has only a weak positive correlation (~0.19) with Stay_Category, which matches the result of the second count plot, suggesting that patients with higher illness severity may stay longer.
# Visitors with Patient shows a moderate negative correlation (~-0.42) with Stay_Category, indicating that patients with more visitors may have shorter hospital stays, possibly due to better support.
# Most other features show low correlation with Stay_Category, suggesting they may have less individual predictive power but could still be useful in combination.

### 1.5.5 Scatter plots with regression lines only for key features strongly correlated with Stay_Category

In [ ]:
#Calculate correlation with Stay_Category
correlations = df_visualization.corr()['Stay_Category'].drop('Stay_Category')
#Identify features with |r|>0,15
key_features = correlations[correlations.abs() > 0.15].index.tolist()
print("Features with |r| > 0.15:\n", key_features)

In [ ]:
from scipy import stats

In [ ]:
for col in key_features:
    plt.figure(figsize=(8, 5))

# Scatter plot with regression lines
    sns.regplot(x=col, y='Stay_Category', data=df_visualization,
                scatter_kws={'alpha':0.5}, line_kws={'color':'red'})

# Calculate correlation and p-value
    r, p = stats.pearsonr(df_visualization[col], df_visualization['Stay_Category'])

    plt.title(f'{col} vs Stay_Category\nr={r:.3f}, p={p:.3e}')
    plt.xlabel(col)
    plt.ylabel('Stay_Category (0=Long, 1=Medium, 2=Short)')  #
    plt.tight_layout()
    plt.show()

## Part 2 · Machine Learning

We tackled the task in two passes:

| Stage | Problem Framing | Why this step? | Key Result |
|-------|-----------------|----------------|------------|
| **1. Baseline — OLS (regression)** | Treat `Stay_Category` as an *ordered numeric* target (0 = Short, 1 = Medium, 2 = Long) and fit an Ordinary Least Squares model. | • Fast check for any linear signal<br>• Coefficients reveal feature influence |  **R² ≈ 0.23** → linear effects explain little variance. |
| **2. Main models — Random Forest & XGBoost (classification)** | Reframe as a *3-class* problem (`Short/Medium/Long`). Tree-based classifiers capture non-linear interactions and handle categorical splits natively. | • Better suited to discrete classes<br>• Robust to mixed feature types |  **Accuracy ≈ 60 %** after Optuna tuning — ~2.5× improvement over the linear baseline. |

**Take-away**

- OLS served as a quick, interpretable probe but confirmed that linear relationships alone are weak. We started with this as a baseline to observe general trends, even though the task is classification.
- Switching to classification with non-linear models delivers much stronger predictive performance while still allowing feature-importance insights (e.g., via Gini or SHAP values).


### 2 · 1 · Linear Regressor — Ordinary Least Squares (OLS)
The heatmap shows limited linearity (highest correlation ~0.54 and others just above 0.1), so we start with a linear regression as a quick baseline method.

In [ ]:
import statsmodels.api as sm
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

#### Model training and testing

In [ ]:
# Model training

# Add constants to the model
x_OLS = sm.add_constant(X_train)

# define the model using the vars with |correlation values to 'Stay' | > 0.1 from heatmap
selected_columns_OLS = [
    'const',
    'Available Extra Rooms in Hospital',
    'Ward_Type_Q', 'Ward_Type_R', 'Ward_Type_S', 'Ward_Type_T', 'Ward_Type_U',
    'Severity of Illness',
    'Visitors with Patient'
]

# initiate the model
model_OLS = sm.OLS(y_train, x_OLS[selected_columns_OLS])

# fit the model
result_OLS = model_OLS.fit()

# get predictions on train data set
y_train_pred = result_OLS.predict(x_OLS[selected_columns_OLS])

# Model testing

# add intercept to X_test
x_test = sm.add_constant(X_test)

# get predictions on test data set
y_pred = result_OLS.predict(x_test[selected_columns_OLS])


#### Model Evaluation - Computing Key Training & Test Metrics: R² and RMSE

In [ ]:
# compute train RMSE
mse_train = mean_squared_error(y_train, y_train_pred )
rmse_train = np.sqrt(mse_train)

# compute test RMSE
mse_test = mean_squared_error(y_test, y_pred)
rmse_test = np.sqrt(mse_test)

# print the summary of the  fitted model
print(f"OLS Model Fitting summary:\n {result_OLS.summary()}")
print()

# print RMSE, R2 respectively on training set and test set
print("Train R²:", r2_score(y_train, y_train_pred))
print("Train RMSE:", rmse_train)
print("Test R²:", r2_score(y_test, y_pred))
print("Test RMSE:", rmse_test)


#### Key Metrics Overview
| Split | R² | RMSE (category-units) |
|-------|----|-----------------------|
| Train | **0.233** | **0.62** |
| Test  | **0.228** | **0.62** |

#### Interpretation
* **Explained variance ≈ 23 %** → linear effects alone are weak.
* **Visitors** ↑ and **Illness Severity** ↑ → longer stays.
* **Ward Types Q/R/S/T** associate with extended LOS.
* **Extra Rooms** ↑ → slightly shorter stays (capacity effect).

* Train ≈ Test scores → no over/under-fit, but overall accuracy low → motivates Non-Linear model, Random Forest & XGBoost.

*Why start with Linear Regressor and why no further metrics here?*
* **Quick sanity-check** – fits in <1 s and shows that linear relation is weak.
* **Transparent** – coefficients & p-values directly reveal which factors move LOS up or down.
* As a summary, OLS is only a preliminary probe guided by the heat-map correlations. Since it shows limited potential, we reserve detailed evaluation for the stronger non-linear models.

### To Rick:
I already explained the linear regressor is only for a quick try-out. You might want to justify about why choosing classifier over regressor.

### 2.2 Random Forest

In [ ]:
# Select features (X) - all columns except missing value indicators and target variables
X = df_one_hot.drop(['Stay', 'Stay_Category', 'Stay_missing', 'Bed_Grade_missing'], axis=1)
# Select target (y)
y = df_one_hot['Stay_Category']

# Update the NumPy array version of X if you need it for subsequent cells
x = X.values

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=0)

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split

# 原始資料切分
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=0)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'random_state': 0
    }

    clf = RandomForestClassifier(**params)
    score = cross_val_score(clf, x_train, y_train, cv=3, scoring='accuracy').mean()
    return score

# 建立 study，開始調整參數
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5)

# 印出最佳參數
print("Best Parameters:", study.best_params)

# 用最佳參數重新訓練模型
best_rf = RandomForestClassifier(**study.best_params)
best_rf.fit(x_train, y_train)

# 預測
y_pred = best_rf.predict(x_test)
'''
[I 2025-07-11 14:56:44,321] A new study created in memory with name: no-name-4736a1b5-0156-4df6-9a36-de5ef06f4b20
[I 2025-07-11 14:57:43,328] Trial 0 finished with value: 0.6039283619824076 and parameters: {'n_estimators': 271, 'max_depth': 13, 'min_samples_split': 2, 'min_samples_leaf': 7, 'max_features': 'log2', 'criterion': 'entropy', 'bootstrap': False}. Best is trial 0 with value: 0.6039283619824076.
[I 2025-07-11 15:04:33,186] Trial 1 finished with value: 0.5693176709630404 and parameters: {'n_estimators': 317, 'max_depth': 19, 'min_samples_split': 15, 'min_samples_leaf': 11, 'max_features': None, 'criterion': 'gini', 'bootstrap': False}. Best is trial 0 with value: 0.6039283619824076.
[I 2025-07-11 15:06:26,825] Trial 2 finished with value: 0.5979617221695005 and parameters: {'n_estimators': 856, 'max_depth': 10, 'min_samples_split': 6, 'min_samples_leaf': 11, 'max_features': 'log2', 'criterion': 'entropy', 'bootstrap': True}. Best is trial 0 with value: 0.6039283619824076.
[I 2025-07-11 15:13:22,548] Trial 3 finished with value: 0.601307218332872 and parameters: {'n_estimators': 494, 'max_depth': 26, 'min_samples_split': 12, 'min_samples_leaf': 18, 'max_features': None, 'criterion': 'entropy', 'bootstrap': True}. Best is trial 0 with value: 0.6039283619824076.
[I 2025-07-11 15:16:04,338] Trial 4 finished with value: 0.6028606439817257 and parameters: {'n_estimators': 807, 'max_depth': 12, 'min_samples_split': 14, 'min_samples_leaf': 3, 'max_features': 'log2', 'criterion': 'entropy', 'bootstrap': False}. Best is trial 0 with value: 0.6039283619824076.
'''

In [ ]:
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier(
    n_estimators=271,
    max_depth=13,
    min_samples_split=2,
    min_samples_leaf=7,
    max_features='log2',
    criterion='entropy',
    bootstrap=False,
    random_state=0
)

classifier.fit(x_train, y_train)


In [ ]:
y_pred = classifier.predict(x_test)

y_pred

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)

cm

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))




In [ ]:
importances = classifier.feature_importances_
# Get the feature names from the X DataFrame
features = X.columns.tolist()

plt.figure(figsize=(10, 8))  # Adjust figure size for better readability with more features
plt.barh(features, importances)
plt.xlabel("Feature Importance")
plt.title("Random Forest Feature Importance")
plt.tight_layout() # Adjust layout to prevent labels overlapping
plt.show()

### 2.3 XGBoost 

In [ ]:
X = df_one_hot.drop(["Stay_Category", "Stay", "Stay_missing", "Bed_Grade_missing"], axis = 1)
Y = df_one_hot["Stay_Category"]
from sklearn.model_selection import train_test_split, cross_val_score
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.25, random_state = 42)

In [ ]:
!pip install xgboost
!pip install optuna
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
# Use Optuna, an automatic hyperparameter optimization framework, to find the best set of parameters for machine learning model
def objective(trial):
    param = {
        'objective': 'multi:softmax',
        'num_class': len(y_train.unique()),
        'eval_metric': 'mlogloss',
        'use_label_encoder': False,
        'random_state': 42,
        
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 2.0),
    }

    model_XGBoost = xgb.XGBClassifier(**param)
    model_XGBoost.fit(x_train, y_train)
    y_pred = model_XGBoost.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

In [ ]:
#Optimization
study = optuna.create_study(direction='maximize')  
study.optimize(objective, n_trials=10)  

In [ ]:
#Use the best parameter combination to predict target
best_params = {
    'n_estimators': 152, 'max_depth': 6, 
    'learning_rate': 0.17688248405678625, 
    'subsample': 0.845905704044682, 
    'colsample_bytree': 0.8380002966457403, 
    'gamma': 0.32589824217309266, 
    'reg_alpha': 0.9963020917620169, 
    'reg_lambda': 0.4904211799171074,
    'objective': 'multi:softmax',
    'num_class': len(y_train.unique()),     
    'eval_metric': 'mlogloss',
    'use_label_encoder': False,
    'random_state': 42
}

model_XGBoost = xgb.XGBClassifier(**best_params)
model_XGBoost.fit(x_train, y_train)
y_pred = model_XGBoost.predict(x_test)

#print out the valuation
print("Accuracy: ", model_XGBoost.score(x_test, y_test) * 100)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nFeature Importance\n")
importance = pd.Series(model_XGBoost.feature_importances_,
                       index = x_train.columns.values)
importance.nlargest(5).plot(kind='barh', figsize = (6,4), title="Top 15 Feature Importances")
plt.xlabel("Feature Importance Score")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

#### Classification Report

- Accuracy: 61%
- Class 2 (longest stay): Best performance with F1-score = 0.67  
- Class 0 (shortest stay): Weakest performance, F1-score = 0.50

| Class | Precision | Recall | F1-score | Support |
|-------|-----------|--------|----------|---------|
| 0     | 0.58      | 0.43   | 0.50 | 30739   |
| 1     | 0.60      | 0.71   | 0.65     | 46145   |
| 2     | 0.68      | 0.66   | 0.67     | 18648   |
| Accuracy     | | |0.61 |95532
| Macro Avg    | 0.62 | 0.60 | 0.61 |95532
| Weighted Avg | 0.61 | 0.61 | 0.60 |95532

#### The top 3 important feature: 
- Visitors with Patient
- Ward_Type_Q
- Ward_Type_S

In [ ]:
# print out the optimization history
from optuna.visualization import plot_optimization_history
plotly_config = {"staticPlot": True}
fig = plot_optimization_history(study)
fig.show(config=plotly_config)